# Small-Cap Bankruptcy Classification with LightGBM

An individual project by Aarav Singhvi using financial panel data, class-imbalance handling, and Optuna tuning.

This notebook runs the documented chronological revision in `train.py`. The original notebook remains unchanged outside this package. Original saved results and newly executed results are distinguished in `README.md` and `results/`.

**Target:** the last observed row of a company labelled failed. This is a retrospective proxy, not a verified bankruptcy date or an established forecasting lead time. Read `DATA.md` before interpreting the model.


## 1. Set up

Install `requirements.txt` and JupyterLab in your Python environment. Download the linked Kaggle CSV and place it in `data/american_bankruptcy.csv`. Open this notebook from the repository root.


In [ ]:
from pathlib import Path
from train import load_data, run_experiment

DATA_PATH = Path("data/american_bankruptcy.csv")
OUTPUT_PATH = Path("artifacts")
N_TRIALS = 30

if not DATA_PATH.exists():
    raise FileNotFoundError("Download american_bankruptcy.csv from the source in DATA.md and put it in data/.")


## 2. Inspect the data

The loader checks the required columns, finite values, duplicate company/year records, and company-level status consistency. Targets are constructed only after explicit company/year sorting.


In [ ]:
data = load_data(DATA_PATH)
print(f"Rows: {len(data):,}; companies: {data.company_name.nunique():,}")
print(f"Years: {data.year.min()}–{data.year.max()}")
print("Retrospective target counts:")
print(data.target.value_counts().sort_index())


## 3. Train and evaluate

Five expanding calendar windows within 1999–2015 evaluate untouched validation rows. Each fold selects its cohort using training history, then fits scaling and SMOTE only on its training partition. All 18 input features are retained to avoid reusing feature choices informed by the full dataset.

Optuna selects parameters by mean validation average precision. A fixed baseline is retained if it scores better. Model selection uses only development data; the selected model is then evaluated on 2016–2018 records of companies chosen from training history. The decision threshold is fixed at 0.5. This holdout period has already been examined in the original project, so it is not a newly collected, pristine external test set.

The run saves metrics, predictions, tuning trials, model weights, scaling parameters, and figures in `artifacts/`.


In [ ]:
metrics = run_experiment(DATA_PATH, OUTPUT_PATH, trials=N_TRIALS)


## 4. Inspect tradeoffs

Read recall together with precision and the confusion matrix. A classifier that always predicts the majority class can have very high accuracy on this dataset. The model's raw scores are not calibrated probabilities, particularly after oversampling.


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT_PATH / "evaluation.png")))
display(Image(filename=str(OUTPUT_PATH / "feature_importance.png")))
print("Always-negative accuracy:", metrics["all_negative_accuracy"])
print("Holdout metrics:", metrics["holdout"])
